# Feature Engineering

В этом ноутбуке мы готовим признаки для моделей. Набор фич будет один и тот же для всех алгоритмов, чтобы потом честно сравнивать качество моделей. Разница между моделями будет только в preprocessing на этапе обучения.

В конце должны получиться два готовых датафрейма: `train_fe` и `test_fe`.

## 1. Setup

Подключаем библиотеки, задаём пути к данным и сохраняем основные константы, которые будем использовать дальше в ноутбуке.

In [72]:
import json
import pickle
import re
import warnings
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

ROOT          = Path(".")
DATA_DIR      = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "target"

# Константы, окна
MAX_RECENCY_SEC = 10 * 365 * 24 * 3600

WINDOWS = {
    "24h": pd.Timedelta(days=1),
    "3d":  pd.Timedelta(days=3),
    "7d":  pd.Timedelta(days=7),
    "30d": pd.Timedelta(days=30),
    "90d": pd.Timedelta(days=30),
}

NON_FEATURE_COLUMNS = {
    "lead_id", "user_id",
    "assignment_ts", "assignment_date",
    TARGET,
}

# числовые по типу, но категориальные по смыслу
MANUAL_CAT = {"assignment_hour", "assignment_weekday", "is_weekend"}

## 2. Загрузка данных

Загружаем train, test и историю событий пользователей. Сразу приводим временные колонки к нужному формату, чтобы потом можно было считать признаки по времени.

In [73]:
train  = pd.read_csv(DATA_DIR / "train.csv",  parse_dates=["assignment_ts"])
test   = pd.read_csv(DATA_DIR / "test.csv",   parse_dates=["assignment_ts"])
events = pd.read_csv(DATA_DIR / "events.csv", parse_dates=["event_ts"])

train["assignment_date"] = pd.to_datetime(train["assignment_date"]).dt.normalize()
test["assignment_date"]  = pd.to_datetime(test["assignment_date"]).dt.normalize()

print(f"train : {train.shape}")
print(f"test  : {test.shape}")
print(f"events: {events.shape}")

train : (13694, 119)
test  : (4306, 118)
events: (254705, 7)


Прежде чем строить новые признаки, заполняем пропуски в оконных колонках. Это нужно, чтобы дальше расчёты были стабильными и не появлялось лишних `NaN`.

In [74]:
def impute_window_columns(train_df: pd.DataFrame, test_df: pd.DataFrame):
    """
    Для колонок вида <base>_<N>d:
    - счётчики: заполняем левым значением, иначе 0
    - непрерывные: заполняем медианой от слева или справа, иначе медиана по всему
    """
    pattern = re.compile(r"^(.*)_(\d+)d$")
    groups: dict[str, dict[int, str]] = {}

    for col in train_df.columns:
        m = pattern.match(col)
        if m:
            base, win = m.group(1), int(m.group(2))
            groups.setdefault(base, {})[win] = col

    for base, wins in groups.items():
        cols = [wins[w] for w in sorted(wins)]
        valid = train_df[cols].dropna()
        is_counter = (valid % 1 == 0).all().all() if not valid.empty else True

        if is_counter:
            train_df[cols] = train_df[cols].ffill(axis=1).fillna(0)
            test_df[cols]  = test_df[cols].ffill(axis=1).fillna(0)
        else:
            train_df[cols] = train_df[cols].interpolate(axis=1).bfill(axis=1).ffill(axis=1)
            test_df[cols]  = test_df[cols].interpolate(axis=1).bfill(axis=1).ffill(axis=1)
            for col in cols:
                median = train_df[col].median()
                train_df[col] = train_df[col].fillna(median)
                test_df[col]  = test_df[col].fillna(median)

    return train_df, test_df


train, test = impute_window_columns(train, test)

print(f"Пропусков в train: {(train.isna().sum() > 0).sum()}")
print(f"Пропусков в test : {(test.isna().sum() > 0).sum()}")

Пропусков в train: 0
Пропусков в test : 0


## 3. Feature Engineering

Дальше строим новые признаки. Сюда входят признаки по времени, отношения между окнами, комбинации категорий и агрегаты по событиям пользователя.

Сначала определяем несколько вспомогательных функций. Они нужны для безопасного деления, работы со временем и расчёта агрегатов без утечки из будущего.

In [75]:
def sanitize_token(x: str) -> str:
    """Нормализует строку для имени фичи."""
    x = str(x).strip().lower()
    x = re.sub(r"[^0-9a-zA-Zа-яА-Я_]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x if x else "unknown"


def safe_div(num, den) -> pd.Series:
    """Безопасное деление: 0/0 - 0, inf - 0, NaN - 0."""
    num = pd.Series(num, copy=False)
    den = pd.Series(den, copy=False).replace(0, np.nan)
    return (num / den).replace([np.inf, -np.inf], np.nan).fillna(0.0)


def make_hour_bin(series: pd.Series) -> pd.Series:
    """Кодировка часа."""
    s   = pd.to_numeric(series, errors="coerce")
    out = pd.Series("NA", index=series.index, dtype="object")
    out[(s >= 0)  & (s <= 5)]  = "night"
    out[(s >= 6)  & (s <= 11)] = "morning"
    out[(s >= 12) & (s <= 16)] = "day"
    out[(s >= 17) & (s <= 21)] = "evening"
    out[(s >= 22) & (s <= 23)] = "late"
    return out.astype(str)


def merge_asof_by(left_df, query_ts, right_df, by, right_ts_col, right_cols,
                  allow_exact_matches=False) -> pd.DataFrame:
    """
    Обертка для pd.merge_asof для безопасного соединения данных "по времени"
    внутри групп (by).
    """
    
    # Приводим параметр by к списку (merge_asof ожидает список колонок)
    if isinstance(by, str):
        by = [by]
        
    if right_df.empty:
        return pd.DataFrame(np.nan, index=left_df.index, columns=right_cols)

    # Берем только группирующие колонки
    left = left_df[by].copy()
    # Добавляем колонку с временной меткой запроса
    left["__qts__"] = pd.to_datetime(query_ts)
    # Добавляем уникальный идентификатор строки,
    # чтобы позже восстановить исходный порядок
    left["__rid__"] = np.arange(len(left_df))
    left = left.sort_values(["__qts__", *by, "__rid__"]).reset_index(drop=True)

    # Оставляем только нужные колонки:
    # группировки + временную колонку + возвращаемые колонки
    use   = list(dict.fromkeys(by + [right_ts_col] + list(right_cols)))
    right = right_df[use].copy()
    right[right_ts_col] = pd.to_datetime(right[right_ts_col])
    right = right.sort_values([right_ts_col, *by]).reset_index(drop=True)

    merged = pd.merge_asof(
        left, right,
        left_on="__qts__", right_on=right_ts_col,
        by=by, direction="backward",
        allow_exact_matches=allow_exact_matches,
    )
    
    # Возвращаем строки в исходный порядок left_df
    out = merged.sort_values("__rid__").reset_index(drop=True)[right_cols].copy()
    out.index = left_df.index
    return out


def window_counts(base_df, ref_df, by, prefix, count_col,
                  windows, base_ts="assignment_ts", ref_ts="event_ts") -> pd.DataFrame:
    """
    Cumulative count - разность = число событий в окне.
    Возвращает DataFrame с колонками prefix_cnt_all и prefix_cnt_<window>.
    """
    
    feats = pd.DataFrame(index=base_df.index)
    
    # Находим последнее событие пользователя строго до base_ts.
    # count_col в ref_df — это накопительный cumcount,
    # который уже посчитан заранее при подготовке ref_df.
    cur = merge_asof_by(
        base_df, base_df[base_ts], ref_df,
        by=by, right_ts_col=ref_ts, right_cols=[count_col],
    )[count_col].fillna(0)

    feats[f"{prefix}_cnt_all"] = cur.astype(float)

    #  cur  = число событий до assignment_ts           (правая граница окна)
    #  past = число событий до assignment_ts - delta   (левая граница окна)
    #  cnt  = cur - past  -  число событий внутри окна

    for wn, delta in windows.items():
        past = merge_asof_by(
            base_df, base_df[base_ts] - delta, ref_df,
            by=by, right_ts_col=ref_ts, right_cols=[count_col],
        )[count_col].fillna(0)
        feats[f"{prefix}_cnt_{wn}"] = (cur - past).clip(lower=0).astype(float)

    return feats


def window_sum_stats(base_df, ref_df, by, prefix,
                     count_col, sum_col, sumsq_col, windows,
                     base_ts="assignment_ts", ref_ts="event_ts",
                     include_all=True) -> pd.DataFrame:
    """
    Mean / std в окне через cumulative sum + sum-of-squares.
    Возвращает DataFrame с колонками mean/std/count для каждого окна.
    """
    # Один merge_asof возвращает сразу три колонки:
    #   count_col — сколько ненулевых цен было
    #   sum_col   — их сумма
    #   sumsq_col — сумма квадратов
    
    feats = pd.DataFrame(index=base_df.index)
    cur = merge_asof_by(
        base_df, base_df[base_ts], ref_df,
        by=by, right_ts_col=ref_ts,
        right_cols=[count_col, sum_col, sumsq_col],
    ).fillna(0)

    if include_all:
        m = safe_div(cur[sum_col], cur[count_col])
        v = (safe_div(cur[sumsq_col], cur[count_col]) - m ** 2).clip(lower=0)
        feats[f"{prefix}_mean_all"]  = m
        feats[f"{prefix}_std_all"]   = np.sqrt(v)
        feats[f"{prefix}_count_all"] = cur[count_col]

    #Также для окна
    for wn, delta in windows.items():
        past = merge_asof_by(
            base_df, base_df[base_ts] - delta, ref_df,
            by=by, right_ts_col=ref_ts,
            right_cols=[count_col, sum_col, sumsq_col],
        ).fillna(0)
        cnt = (cur[count_col] - past[count_col]).clip(lower=0)
        sm  = cur[sum_col]    - past[sum_col]
        sq  = cur[sumsq_col]  - past[sumsq_col]
        m   = safe_div(sm, cnt)
        v   = (safe_div(sq, cnt) - m ** 2).clip(lower=0)
        feats[f"{prefix}_count_{wn}"] = cnt
        feats[f"{prefix}_mean_{wn}"]  = m
        feats[f"{prefix}_std_{wn}"]   = np.sqrt(v)

    return feats

## 3.1 Time features

Добавляем простые признаки времени назначения: час, день недели и их циклическое представление через `sin/cos`. Так модель лучше воспринимает время как повторяющийся цикл.

In [76]:
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Циклическое кодирование времени назначения.
    """
    df  = df.copy()
    ts  = pd.to_datetime(df["assignment_ts"], errors="coerce")
    h   = pd.to_numeric(df["assignment_hour"],    errors="coerce")
    wd  = pd.to_numeric(df["assignment_weekday"], errors="coerce")

    df["hour_sin"]                = np.sin(2 * np.pi * h  / 24)
    df["hour_cos"]                = np.cos(2 * np.pi * h  / 24)
    df["weekday_sin"]             = np.sin(2 * np.pi * wd / 7)
    df["weekday_cos"]             = np.cos(2 * np.pi * wd / 7)
    df["minutes_since_day_start"] = ts.dt.hour * 60 + ts.dt.minute

    return df

## 3.2 Ratios и momentum

Здесь считаем отношения между короткими и длинными окнами. Также добавляем признаки по прошлым назначениям, пользовательской активности и простые производные метрики.

In [77]:
def add_ratio_and_momentum_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    отношения по всем колонкам вида <base>_<N>d.
    Дополнительно: признаки по прошлым назначениям, пользовательской активности и простые производные метрики.
    """
    df      = df.copy()
    pattern = re.compile(r"^(.*)_(\d+)d$")
    groups: dict[str, list[int]] = {}

    for col in df.columns:
        m = pattern.match(col)
        if m:
            groups.setdefault(m.group(1), []).append(int(m.group(2)))

    # momentum ratios для всех пар окон
    for base, wins in groups.items():
        for w_s, w_l in combinations(sorted(wins), 2):
            df[f"{base}_{w_s}d_over_{w_l}d"] = safe_div(
                df[f"{base}_{w_s}d"], df[f"{base}_{w_l}d"]
            )

    def common_wins(*bases):
        sets = [set(groups.get(b, [])) for b in bases]
        return sorted(set.intersection(*sets))

    # воронка leadgen
    for w in common_wins("leadgen_prev_assigned", "leadgen_prev_answered", "leadgen_prev_positive"):
        ass = df[f"leadgen_prev_assigned_{w}d"]
        ans = df[f"leadgen_prev_answered_{w}d"]
        pos = df[f"leadgen_prev_positive_{w}d"]
        df[f"leadgen_answer_rate_{w}d"]   = safe_div(ans, ass)
        df[f"leadgen_positive_rate_{w}d"] = safe_div(pos, ass)
        df[f"leadgen_pos_given_ans_{w}d"] = safe_div(pos, ans)

    # пользовательской активности
    for num_b, den_b, name in [
        ("item_favorites", "item_views",    "fav_per_view"),
        ("detail_expands", "item_views",    "expand_per_view"),
        ("chat_opens",     "user_contacts", "chat_per_contact"),
        ("call_clicks",    "user_contacts", "call_per_contact"),
    ]:
        for w in common_wins(num_b, den_b):
            df[f"{name}_{w}d"] = safe_div(df[f"{num_b}_{w}d"], df[f"{den_b}_{w}d"])

    # простые производные метрики
    if "prior_assignments_30d" in df.columns and "user_active_days_30d" in df.columns:
        df["assignments_per_active_day"] = safe_div(
            df["prior_assignments_30d"], df["user_active_days_30d"]
        )
    if "mileage_km_log" in df.columns and "car_age_years" in df.columns:
        df["mileage_per_year"] = safe_div(df["mileage_km_log"], df["car_age_years"])

    return df

## 3.3 Кросс-признаки

В этом блоке создаём комбинации категориальных признаков. Такие кроссы помогают модели учитывать контекст, например не просто источник лида или регион по отдельности, а их сочетание (более полезно для линейных моделей, для деревьев избыточно, т.к. они сами могут уловить контекст)

In [78]:
def add_cross_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Категориальные кроссы (пары) и создание кат. фич времени
    """
    df = df.copy()

    if "assignment_hour" in df.columns:
        df["assignment_hour_bin"] = make_hour_bin(df["assignment_hour"])

    pairs = [
        ("lead_source",  "region"),
        ("lead_source",  "call_center"),
        ("lead_channel", "car_segment"),
        ("car_segment",  "price_bucket"),
        ("price_bucket", "region"),
        ("region",       "is_weekend"),
        ("call_center",  "car_segment"),
        ("lead_source",  "assignment_hour_bin"),
        ("call_center",  "assignment_hour_bin"),
        ("region",       "assignment_hour_bin"),
        ("car_segment",  "assignment_hour_bin"),
    ]
    for c1, c2 in pairs:
        if c1 in df.columns and c2 in df.columns:
            df[f"{c1}_x_{c2}"] = df[c1].astype(str) + "__" + df[c2].astype(str)

    return df

## 3.4 История событий пользователя

Теперь строим признаки из таблицы `events`. Считаем, когда было последнее событие, сколько событий было за разные промежутки времени, какие типы событий чаще встречались и насколько поведение пользователя было активным и разнообразным.

In [79]:
def add_event_features(base_df: pd.DataFrame, events_df: pd.DataFrame) -> pd.DataFrame:
    """
    Создание доп. признаков из events
    """
    df = base_df.copy()
    ev = events_df.copy()

    ev["event_ts"] = pd.to_datetime(ev["event_ts"], errors="coerce")
    ev = ev.loc[ev["event_ts"].notna()].copy()

    # Заполняем пропуски если есть
    ev["event_type"]  = ev.get("event_type",  pd.Series("NA", index=ev.index)).fillna("NA").astype(str)
    ev["src_slot"]    = ev.get("src_slot",     pd.Series("NA", index=ev.index)).fillna("NA").astype(str)
    ev["ctx_seq_str"] = ev.get("ctx_seq",      pd.Series("NA", index=ev.index)).fillna("NA").astype(str)
    ev["item_price_log"] = pd.to_numeric(ev.get("item_price_log"), errors="coerce")

    # Сортируем по id и времени и считаем счетчик (cumcount) и тип предыдущего события
    ev_u = ev.sort_values(["user_id", "event_ts"]).copy()
    ev_u["eidx"]   = ev_u.groupby("user_id").cumcount() + 1
    ev_u["prev_et"] = ev_u.groupby("user_id")["event_type"].shift(1).fillna("NONE")

    # Кумулятивные значения для цен (сумма валидных/всего и квадраты)
    ev_u["pv"]   = ev_u["item_price_log"].notna().astype(int)
    ev_u["pf"]   = ev_u["item_price_log"].fillna(0.0)
    ev_u["psqf"] = ev_u["pf"] ** 2
    ev_u["pc"]   = ev_u.groupby("user_id")["pv"].cumsum()
    ev_u["ps"]   = ev_u.groupby("user_id")["pf"].cumsum()
    ev_u["psq"]  = ev_u.groupby("user_id")["psqf"].cumsum()

    # Накопительный счётчик активных дней
    ev_u["event_day"]         = ev_u["event_ts"].dt.floor("D")
    prev_day                  = ev_u.groupby("user_id")["event_day"].shift(1)
    ev_u["is_new_day"]        = (ev_u["event_day"] != prev_day).astype(int)
    ev_u["active_day_cum"]    = ev_u.groupby("user_id")["is_new_day"].cumsum()

    # Накопительное число уникальных src_slot и ctx_seq
    ev_u["is_first_src"] = (ev_u.groupby(["user_id", "src_slot"]).cumcount() == 0).astype(int)
    ev_u["src_nunique_cum"] = ev_u.groupby("user_id")["is_first_src"].cumsum()

    ev_u["is_first_ctx"]    = (ev_u.groupby(["user_id", "ctx_seq_str"]).cumcount() == 0).astype(int)
    ev_u["ctx_nunique_cum"] = ev_u.groupby("user_id")["is_first_ctx"].cumsum()

    # Накопительная статистика по timestamps (для вычисления давности событий)
    ev_u["ts_sec"]        = ev_u["event_ts"].astype("int64") / 1e9
    ev_u["ts_sec_sq"]     = ev_u["ts_sec"] ** 2
    ev_u["ts_sec_cum"]    = ev_u.groupby("user_id")["ts_sec"].cumsum()
    ev_u["ts_sec_sq_cum"] = ev_u.groupby("user_id")["ts_sec_sq"].cumsum()
    ev_u["first_event_ts"] = ev_u.groupby("user_id")["event_ts"].transform("first")

    # Накопительная статистика по gap между событиями
    # gap = время между соседними событиями одного пользователя.
    ev_u["gap_sec"]     = ev_u.groupby("user_id")["event_ts"].diff().dt.total_seconds()
    ev_u["gap_flag"]    = ev_u["gap_sec"].notna().astype(int)
    ev_u["gap_fill"]    = ev_u["gap_sec"].fillna(0.0)
    ev_u["gap_sq_fill"] = ev_u["gap_fill"] ** 2
    # Накопительные счётчик, сумма, сумма квадратов для mean/std,min/max
    ev_u["gap_cnt_cum"] = ev_u.groupby("user_id")["gap_flag"].cumsum()
    ev_u["gap_sum_cum"] = ev_u.groupby("user_id")["gap_fill"].cumsum()
    ev_u["gap_sq_cum"]  = ev_u.groupby("user_id")["gap_sq_fill"].cumsum()
    ev_u["gap_min_cum"] = (
        ev_u["gap_sec"].fillna(np.inf)
        .groupby(ev_u["user_id"]).cummin()
        .replace(np.inf, np.nan)
    )
    ev_u["gap_max_cum"] = (
        ev_u["gap_sec"].fillna(-np.inf)
        .groupby(ev_u["user_id"]).cummax()
        .replace(-np.inf, np.nan)
    )

    # последнее событие до assignment_ts
    ul = merge_asof_by(
        df, df["assignment_ts"], ev_u,
        by=["user_id"], right_ts_col="event_ts",
        right_cols=[
            "event_ts", "event_type", "prev_et",
            "eidx", "item_price_log", "pc", "ps", "psq",
            "src_slot", "ctx_seq_str",
            "active_day_cum", "src_nunique_cum", "ctx_nunique_cum",
            "ts_sec_cum", "ts_sec_sq_cum", "first_event_ts",
            "gap_cnt_cum", "gap_sum_cum", "gap_sq_cum",
            "gap_min_cum", "gap_max_cum",
        ],
        allow_exact_matches=False,
    )

    # базовые last-event фичи
    df["last_event_type"] = ul["event_type"].fillna("NONE").astype(str)
    df["prev_event_type"] = ul["prev_et"].fillna("NONE").astype(str)
    df["last_src_slot"]   = ul["src_slot"].fillna("NA").astype(str)
    df["last_ctx_seq"]    = ul["ctx_seq_str"].fillna("NA").astype(str)
    df["user_last_event_idx"] = ul["eidx"].fillna(0)
    # Сколько секунд прошло с последнего события до assignment
    df["sec_since_last_event"] = (
        (df["assignment_ts"] - ul["event_ts"]).dt.total_seconds()
    ).fillna(MAX_RECENCY_SEC)
    # Цена последнего события и разница с текущей ценой объявления
    df["last_event_price"]          = ul["item_price_log"]
    df["price_diff_last"]           = df["item_price_log"] - ul["item_price_log"]
    df["user_event_price_abs_diff_last"] = df["price_diff_last"].abs()
    # Средняя цена по всей истории пользователя и отклонение от неё
    pm = safe_div(ul["ps"].fillna(0), ul["pc"].fillna(0))
    df["user_price_mean_all"]  = pm
    df["price_diff_user_mean"] = df["item_price_log"] - pm

    # summary-фичи за всю историю
    df["user_event_active_days_all"]    = ul["active_day_cum"].fillna(0).astype(float)
    df["user_event_src_slot_nunique_all"] = ul["src_nunique_cum"].fillna(0).astype(float)
    df["user_event_ctx_nunique_all"]    = ul["ctx_nunique_cum"].fillna(0).astype(float)

    # статистики "возраста" событий
    ts_parsed   = pd.to_datetime(df["assignment_ts"], errors="coerce")
    ts_sec_asgn = ts_parsed.astype("int64").where(ts_parsed.notna(), np.nan) / 1e9
    n_all        = ul["eidx"].fillna(0)

    mean_age_sec = safe_div(
        ts_sec_asgn * n_all - ul["ts_sec_cum"].fillna(0), n_all
    )
    mean_ts_sec  = safe_div(ul["ts_sec_cum"].fillna(0), n_all)
    var_ts_sec   = (
        safe_div(ul["ts_sec_sq_cum"].fillna(0), n_all) - mean_ts_sec ** 2
    ).clip(lower=0)

    df["user_event_age_h_mean_all"] = (mean_age_sec / 3600).where(n_all > 0, np.nan)
    df["user_event_age_h_std_all"]  = (np.sqrt(var_ts_sec) / 3600).where(n_all > 0, np.nan)
    df["user_event_age_h_max_all"]  = (
        (ts_parsed - ul["first_event_ts"]).dt.total_seconds() / 3600
    ).where(n_all > 0, np.nan)

    # статистики gap между событиями
    gap_cnt      = ul["gap_cnt_cum"].fillna(0)
    gap_mean_sec = safe_div(ul["gap_sum_cum"].fillna(0), gap_cnt)
    gap_var_sec  = (
        safe_div(ul["gap_sq_cum"].fillna(0), gap_cnt) - gap_mean_sec ** 2
    ).clip(lower=0)

    df["user_event_gap_h_mean_all"] = (gap_mean_sec / 3600).where(gap_cnt > 0, np.nan)
    df["user_event_gap_h_std_all"]  = (np.sqrt(gap_var_sec) / 3600).where(gap_cnt > 0, np.nan)
    df["user_event_gap_h_min_all"]  = (ul["gap_min_cum"] / 3600).where(gap_cnt > 0, np.nan)
    df["user_event_gap_h_max_all"]  = (ul["gap_max_cum"] / 3600).where(gap_cnt > 0, np.nan)

    # количество событий пользователя в временных окнах
    user_cnt = window_counts(
        df, ev_u[["user_id", "event_ts", "eidx"]],
        by=["user_id"], prefix="user_events", count_col="eidx",
        windows=WINDOWS,
    )
    df = pd.concat([df, user_cnt], axis=1)

    # статистики цен в окнах 7d и 30d
    price_feats = window_sum_stats(
        df, ev_u[["user_id", "event_ts", "pc", "ps", "psq"]],
        by=["user_id"], prefix="user_event_price",
        count_col="pc", sum_col="ps", sumsq_col="psq",
        windows={"7d": pd.Timedelta(days=7), "30d": pd.Timedelta(days=30)},
        include_all=False,
    )
    df = pd.concat([df, price_feats], axis=1)

    for sfx in ("7d", "30d"):
        col = f"user_event_price_mean_{sfx}"
        if col in df.columns:
            df[f"price_diff_user_mean_{sfx}"] = df["item_price_log"] - df[col]

    # фичи по каждому типу события
    event_types = sorted(ev["event_type"].dropna().unique())
    event_slugs = []

    for et in event_types:
        slug = sanitize_token(et)
        event_slugs.append(slug)

        ev_t = (
            ev_u.loc[ev_u["event_type"] == et, ["user_id", "event_ts"]]
            .sort_values(["user_id", "event_ts"])
            .reset_index(drop=True)
        )
        if ev_t.empty:
            continue
        ev_t["tidx"] = ev_t.groupby("user_id").cumcount() + 1

        lt  = merge_asof_by(df, df["assignment_ts"], ev_t,
                            by=["user_id"], right_ts_col="event_ts",
                            right_cols=["event_ts", "tidx"])
        cur = lt["tidx"].fillna(0)

        df[f"evt_{slug}_cnt_all"]  = cur.astype(float)
        df[f"sec_since_{slug}"]    = (
            (df["assignment_ts"] - lt["event_ts"]).dt.total_seconds()
        ).fillna(MAX_RECENCY_SEC)

        for wn, delta in WINDOWS.items():
            past = merge_asof_by(
                df, df["assignment_ts"] - delta, ev_t,
                by=["user_id"], right_ts_col="event_ts", right_cols=["tidx"],
            )["tidx"].fillna(0)

            cnt   = (cur - past).clip(lower=0)
            total = df.get(f"user_events_cnt_{wn}", pd.Series(0, index=df.index))

            df[f"evt_{slug}_cnt_{wn}"]   = cnt
            df[f"evt_{slug}_has_{wn}"]   = (cnt > 0).astype("int8")
            df[f"evt_{slug}_share_{wn}"] = safe_div(cnt, total)

    # разнообразие типов событий
    # Сколько разных типов событий было у пользователя за всю историю
    cnt_all_cols = [f"evt_{s}_cnt_all" for s in event_slugs if f"evt_{s}_cnt_all" in df.columns]
    df["user_event_type_nunique_all"] = (
        (df[cnt_all_cols] > 0).sum(axis=1).astype(float) if cnt_all_cols else 0.0
    )

    # Разнообразие и энтропия типов событий в каждом окне
    for wn in WINDOWS:
        has_cols   = [f"evt_{s}_has_{wn}"   for s in event_slugs if f"evt_{s}_has_{wn}"   in df.columns]
        share_cols = [f"evt_{s}_share_{wn}" for s in event_slugs if f"evt_{s}_share_{wn}" in df.columns]

        if has_cols:
            df[f"user_event_type_nunique_{wn}"] = df[has_cols].sum(axis=1)
        if share_cols:
            p = df[share_cols].clip(lower=0)
            df[f"user_event_type_entropy_{wn}"]   = -(p * np.log(p.clip(lower=1e-12))).sum(axis=1)
            df[f"user_event_type_top_share_{wn}"] = p.max(axis=1)

    # соотношение активности между окнами
    for s, l in [("1h", "24h"), ("6h", "24h"), ("24h", "7d")]:
        c1, c2 = f"user_events_cnt_{s}", f"user_events_cnt_{l}"
        if c1 in df.columns and c2 in df.columns:
            df[f"user_events_cnt_{s}_over_{l}"] = safe_div(df[c1], df[c2])

    # фичи по контекстам (ctx_seq)
    ctx_windows = {
        "6h":  pd.Timedelta(hours=6),
        "24h": pd.Timedelta(hours=24),
        "3d":  pd.Timedelta(days=3),
        "7d":  pd.Timedelta(days=7),
    }

    for ctx in sorted(ev["ctx_seq_str"].dropna().unique()):
        slug = sanitize_token(ctx)

        ev_c = (
            ev_u.loc[ev_u["ctx_seq_str"] == ctx, ["user_id", "event_ts"]]
            .sort_values(["user_id", "event_ts"])
            .reset_index(drop=True)
        )
        if ev_c.empty:
            continue
        ev_c["cidx"] = ev_c.groupby("user_id").cumcount() + 1

        lc  = merge_asof_by(df, df["assignment_ts"], ev_c,
                            by=["user_id"], right_ts_col="event_ts",
                            right_cols=["cidx"])
        cur = lc["cidx"].fillna(0)
        df[f"ctx_{slug}_cnt_all"] = cur.astype(float)

        for wn, delta in ctx_windows.items():
            past = merge_asof_by(
                df, df["assignment_ts"] - delta, ev_c,
                by=["user_id"], right_ts_col="event_ts", right_cols=["cidx"],
            )["cidx"].fillna(0)
            df[f"ctx_{slug}_cnt_{wn}"] = (cur - past).clip(lower=0).astype(float)

    # заполняем NaN нулями в count-колонках
    fill_cols = (
        ["user_event_type_nunique_all",
         "user_event_active_days_all",
         "user_event_src_slot_nunique_all",
         "user_event_ctx_nunique_all"]
        + [c for c in df.columns if c.startswith("evt_") and "_cnt_" in c]
        + [c for c in df.columns if c.startswith("ctx_") and "_cnt_" in c]
    )
    for c in fill_cols:
        if c in df.columns:
            df[c] = df[c].fillna(0)

    return df

## 3.5 Воронка и глубина событий

На основе событий добавляем более понятные поведенческие признаки. Отдельно смотрим поверхностные действия, более глубокий интерес и контактные события. Это помогает лучше описать силу намерения пользователя.

In [80]:
def add_funnel_and_depth_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Воронка событий:
    - глубина последнего события (stage, depth, is_shallow/deep/contact)
    - shallow/deep/contact counts, shares, flags в окнах 
    - funnel ratios: fav/view, contact/view, deep/shallow
    - max_event_stage в окне
    - recency последнего deep / contact события
    - флаги shallow_last_but_deep_24h / shallow_last_but_contact_24h
    """
    df = df.copy()

    # Числовой уровень воронки для каждого типа события
    STAGE = {
        "search":     0,   # верхний уровень — просто искал
        "item_view":  1,   # посмотрел объявление
        "favorite":   2,   # добавил в избранное
        "chat_open":  3,   # открыл чат — контакт с продавцом
        "call_click": 3,   # нажал "позвонить" — тоже контакт
    }
    DEPTH = {
        "search": "search", "item_view": "view",
        "favorite": "favorite", "chat_open": "contact", "call_click": "contact",
    }

    def gcol(name):
        return df[name] if name in df.columns else pd.Series(0.0, index=df.index)
    # Некоторые признаки уже были посчитаны, поэтому мы просто обогощаем другими
    # Признаки последнего события пользователя
    if "last_event_type" in df.columns:
        df["last_event_stage"] = df["last_event_type"].map(STAGE).fillna(-1).astype(float)
        df["last_event_depth"] = df["last_event_type"].map(DEPTH).fillna("other").astype(str)
        df["last_is_shallow"]  = df["last_event_type"].isin(["search", "item_view"]).astype("int8")
        df["last_is_deep"]     = df["last_event_type"].isin(["favorite", "chat_open", "call_click"]).astype("int8")
        df["last_is_contact"]  = df["last_event_type"].isin(["chat_open", "call_click"]).astype("int8")

    # агрегаты по воронке в каждом временном окне
    for wn in ["1h", "3h", "24h", "7d","30d"]:
        search  = gcol(f"evt_search_cnt_{wn}")
        view    = gcol(f"evt_item_view_cnt_{wn}")
        fav     = gcol(f"evt_favorite_cnt_{wn}")
        contact = gcol(f"evt_chat_open_cnt_{wn}") + gcol(f"evt_call_click_cnt_{wn}")
        deep    = fav + contact
        shallow = search + view
        total   = gcol(f"user_events_cnt_{wn}")

        # Абсолютные счётчики по группам глубины
        df[f"shallow_evt_cnt_{wn}"]  = shallow
        df[f"deep_evt_cnt_{wn}"]     = deep
        df[f"contact_evt_cnt_{wn}"]  = contact
        # Флаги наличия хотя бы одного события данной глубины в окне
        df[f"has_deep_event_{wn}"]   = (deep    > 0).astype("int8")
        df[f"has_contact_event_{wn}"] = (contact > 0).astype("int8")
        # Доли событий каждой группы от общего числа событий в окне
        df[f"shallow_share_{wn}"]    = safe_div(shallow, total)
        df[f"deep_share_{wn}"]       = safe_div(deep,    total)
        df[f"contact_share_{wn}"]    = safe_div(contact, total)
        
        # Конверсии между уровнями воронки
        df[f"item_view_over_search_{wn}"]   = safe_div(view,    search)
        df[f"favorite_over_item_view_{wn}"] = safe_div(fav,     view)
        df[f"contact_over_item_view_{wn}"]  = safe_div(contact, view)
        df[f"deep_over_shallow_{wn}"]       = safe_div(deep,    shallow)

        stage = pd.Series(-1.0, index=df.index)
        for val, cond in [(0, search > 0), (1, view > 0), (2, fav > 0), (3, contact > 0)]:
            stage = stage.where(~cond, float(val))
        df[f"max_event_stage_{wn}"] = stage

    # Максимальный уровень воронки в окне
    for name, cols in [
        ("sec_since_last_contact_event", ["sec_since_chat_open", "sec_since_call_click"]),
        ("sec_since_last_deep_event",    ["sec_since_favorite",  "sec_since_chat_open", "sec_since_call_click"]),
    ]:
        present = [c for c in cols if c in df.columns]
        if present:
            df[name] = df[present].min(axis=1)

    # Пользователь в целом активен на глубоком уровне,
    # но его последнее конкретное событие было поверхностным.
    if "last_is_shallow" in df.columns:
        for sfx, flag in [("deep_24h", "has_deep_event_24h"), ("contact_24h", "has_contact_event_24h")]:
            if flag in df.columns:
                df[f"shallow_last_but_{sfx}"] = (
                    (df["last_is_shallow"] == 1) & (df[flag] == 1)
                ).astype("int8")

    return df

## 4. Сборка всех признаков

Объединяем все шаги в одну функцию, чтобы одинаково применять feature engineering и к train, и к test.

In [81]:
def build_features(base_df: pd.DataFrame, events_df: pd.DataFrame) -> pd.DataFrame:
    df = base_df.copy()
    df = add_time_features(df)
    df = add_ratio_and_momentum_features(df)
    df = add_cross_features(df)
    df = add_event_features(df, events_df)
    df = add_funnel_and_depth_features(df)
    return df


print("Building train features")
train_fe = build_features(train, events)

print("Building test features")
test_fe  = build_features(test, events)

for df_ in (train_fe, test_fe):
    for c in df_.columns:
        if df_[c].dtype == "object" or c in MANUAL_CAT:
            df_[c] = df_[c].fillna("NA").astype(str)


print(f"train_fe: {train_fe.shape}")
print(f"test_fe : {test_fe.shape}")

Building train features
Building test features
train_fe: (13694, 671)
test_fe : (4306, 670)




Перед сохранением ещё раз проверяем пропуски, отдельно обрабатываем числовые колонки и собираем списки категориальных и числовых признаков.

In [82]:
# Столбцы с пропусками 
cols_with_na = train_fe.columns[train_fe.isna().any()].tolist()

for c in train_fe.columns:
    if c not in test_fe.columns:
        continue
    if train_fe[c].dtype != "object":
        med = train_fe[c].median()
        train_fe[c] = train_fe[c].fillna(med)
        test_fe[c]  = test_fe[c].fillna(med)


## 7. Сохранение

Сохраняем готовые датафреймы с признаками и метаинформацию по колонкам. Дальше эти файлы будут использоваться уже в ноутбуке с обучением моделей.

In [83]:
feature_columns = train_fe.columns.tolist()

cat_features = [c for c in feature_columns if train_fe[c].dtype == "object"]
num_features = [c for c in feature_columns if c not in cat_features]

print(f"Final train_fe: {train_fe.shape}")
print(f"Final test_fe : {test_fe.shape}")

Final train_fe: (13694, 671)
Final test_fe : (4306, 670)


In [84]:
train_fe.to_parquet(PROCESSED_DIR / "train_fe.parquet", index=False)
test_fe.to_parquet( PROCESSED_DIR / "test_fe.parquet",  index=False)

with open(PROCESSED_DIR / "feature_meta.json", "w") as f:
    json.dump({
        "feature_columns": feature_columns,
        "cat_features":    cat_features,
        "num_features":    num_features,
    }, f, indent=2, ensure_ascii=False)


print(f"Saved → {PROCESSED_DIR.resolve()}")

Saved → C:\Users\denis\OneDrive\Рабочий стол\avito\data\processed


## 8. Итоги

В результате мы получили единый набор признаков для всех моделей. Дальше на этапе обучения останется только сделать нужный preprocessing для каждого алгоритма и сравнить качество.